# Prototypical meta-learning across emotion datasets

This notebook prepares GoEmotions, MYBully, and CMED in a shared Ekman-style label space for episodic training. `fear` is excluded from every dataset before episodes are formed. The workflow follows the configuration, reproducibility, and reporting style of the dataset notebooks.

In [1]:
"""Imports and runtime diagnostics."""

import json
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.5.1+cu121
CUDA available: True


In [2]:
"""Configuration and reproducibility setup."""

MODEL_NAME = "xlm-roberta-base"
GOEMOTIONS_CHECKPOINT = "best_goemotion_focal_model.pt"
CMED_CHECKPOINT = "best_cmed_focal_model.pt"
GOEMOTIONS_TRAIN_PATH = "db/goemotions/train.tsv"
GOEMOTIONS_VALID_PATH = "db/goemotions/dev.tsv"
GOEMOTIONS_TEST_PATH = "db/goemotions/test.tsv"
GOEMOTIONS_EMOTIONS_PATH = "db/goemotions/emotions.txt"
GOEMOTIONS_EKMAN_PATH = "db/goemotions/ekman_mapping.json"
MYBULLY_VALID_PATH = "hf://datasets/mohanrj/MYBully/val.csv"
CMED_DATA_PATH = "hf://datasets/Johnson8187/Chinese_Multi-Emotion_Dialogue_Dataset/data.csv"
MAX_LENGTH = 140
BATCH_SIZE = 64
EPOCHS = 5
TRAIN_EPISODES_PER_DATASET = 500
VALIDATION_EPISODES_PER_DATASET = 100
MYBULLY_TEST_EPISODES = 200
LEARNING_RATE = 2e-5
RANDOM_SEED = 42
EXCLUDED_CATEGORIES = {"fear"}
CATEGORY_LABELS = ["anger", "disgust", "joy", "sadness", "surprise", "neutral"]
category2id = {label: index for index, label in enumerate(CATEGORY_LABELS)}
id2category = {index: label for label, index in category2id.items()}
NUM_LABELS = len(CATEGORY_LABELS)
NUM_GPUS = min(2, torch.cuda.device_count())
DEVICE = torch.device("cuda:0" if NUM_GPUS else "cpu")
print("Using device:", DEVICE, "| GPUs used:", NUM_GPUS)
USE_AMP = NUM_GPUS > 0
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
if NUM_GPUS:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for gpu_id in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(gpu_id)
        print(f"GPU {gpu_id}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

Using device: cuda:0 | GPUs used: 2


In [3]:
"""Shared label normalization and dataset preparation helpers."""

with open(GOEMOTIONS_EMOTIONS_PATH, encoding="utf-8") as f:
    goemotions_labels = [line.strip() for line in f if line.strip()]
with open(GOEMOTIONS_EKMAN_PATH, encoding="utf-8") as f:
    ekman_mapping = json.load(f)

goemotion_to_category = {emotion: category.lower()
                         for category, emotions in ekman_mapping.items()
                         for emotion in emotions}
goemotion_to_category["neutral"] = "neutral"

# Dataset-native names are mapped where they have a clear shared category.
# Unmapped labels are reported and omitted instead of silently assigned.
native_label_aliases = {
    "anger": "anger", "angry": "anger", "anger tone": "anger",
    "disgust": "disgust", "disgust tone": "disgust",
    "joy": "joy", "happy": "joy", "happiness": "joy", "happily": "joy",
    "love": "joy", "positive": "joy",
    "sad": "sadness", "sadness": "sadness", "sad tone": "sadness",
    "surprise": "surprise", "surprised": "surprise", "surprise tone": "surprise",
    "neutral": "neutral", "neutral tone": "neutral",
    "fear": "fear", "fear tone": "fear",
}

CMED_EMOTION_MAPPING = {
    "平淡語氣": "neutral", "開心語調": "joy",
    "悲傷語調": "sadness", "憤怒語調": "anger",
    "驚訝語調": "surprise", "驚奇語調": "surprise",
    "恐懼語調": "fear", "厭惡語調": "disgust",
    "疑問語調": "surprise",
}

# Rebuild CMED labels from code points to avoid source-file encoding ambiguity.
CMED_EMOTION_MAPPING = {
    "".join(chr(c) for c in (0x5e73, 0x6de1, 0x8a9e, 0x6c23)): "neutral",
    "".join(chr(c) for c in (0x958b, 0x5fc3, 0x8a9e, 0x8abf)): "joy",
    "".join(chr(c) for c in (0x60b2, 0x50b7, 0x8a9e, 0x8abf)): "sadness",
    "".join(chr(c) for c in (0x61a4, 0x6012, 0x8a9e, 0x8abf)): "anger",
    "".join(chr(c) for c in (0x9a5a, 0x8a1d, 0x8a9e, 0x8abf)): "surprise",
    "".join(chr(c) for c in (0x9a5a, 0x5947, 0x8a9e, 0x8abf)): "surprise",
    "".join(chr(c) for c in (0x6050, 0x61fc, 0x8a9e, 0x8abf)): "fear",
    "".join(chr(c) for c in (0x53ad, 0x60e1, 0x8a9e, 0x8abf)): "disgust",
    "".join(chr(c) for c in (0x7591, 0x554f, 0x8a9e, 0x8abf)): "surprise",
}

def normalize_native_label(label):
    value = str(label).strip()
    return CMED_EMOTION_MAPPING.get(value, native_label_aliases.get(value.lower()))

def prepare_frame(frame, text_column, label_column, label_mapper, dataset_name):
    frame = frame[[text_column, label_column]].dropna().copy()
    frame["text"] = frame[text_column].astype(str).str.strip()
    frame["category"] = frame[label_column].map(label_mapper)
    unmapped = sorted(frame.loc[frame["category"].isna(), label_column].astype(str).unique())
    if unmapped:
        print(f"{dataset_name}: unmapped source labels omitted: {unmapped}")
    frame = frame.dropna(subset=["category"]).copy()
    before = len(frame)
    frame = frame[~frame["category"].isin(EXCLUDED_CATEGORIES)].copy()
    print(f"{dataset_name}: removed {before - len(frame)} fear examples")
    frame = frame[frame["category"].isin(CATEGORY_LABELS)].copy()
    frame["label_id"] = frame["category"].map(category2id).astype(int)
    frame["dataset"] = dataset_name
    return frame[["text", "category", "label_id", "dataset"]].reset_index(drop=True)

In [4]:
"""Load the three datasets and build fear-free train/validation/test splits."""

goemotion_raw = {}
for split, path in {"train": GOEMOTIONS_TRAIN_PATH, "validation": GOEMOTIONS_VALID_PATH, "test": GOEMOTIONS_TEST_PATH}.items():
    goemotion_raw[split] = pd.read_csv(path, sep="\t", header=None, names=["text", "emotion", "userid"])

def map_goemotions_label(value):
    # GoEmotions can contain multi-label rows; use the first label as in goemotion.ipynb.
    first_index = int(str(value).split(",")[0])
    if first_index < 0 or first_index >= len(goemotions_labels):
        return None
    return goemotion_to_category.get(goemotions_labels[first_index])

goemotion_splits = {split: prepare_frame(df, "text", "emotion", map_goemotions_label, f"GoEmotions/{split}")
                    for split, df in goemotion_raw.items()}

# MYBully is held out entirely and loaded only for the final test.
mybully_valid_raw = pd.read_csv(MYBULLY_VALID_PATH, usecols=["Tweet", "Emotion"])
mybully_test = prepare_frame(mybully_valid_raw, "Tweet", "Emotion", normalize_native_label, "MYBully/final_test")

# CMED mapping and stratified 80/10/10 split follow cmed.ipynb.
cmed_raw = pd.read_csv(CMED_DATA_PATH, usecols=["text", "emotion"])
cmed = prepare_frame(cmed_raw, "text", "emotion", normalize_native_label, "CMED")
cmed_train, cmed_remaining = train_test_split(cmed, test_size=0.2, random_state=RANDOM_SEED, stratify=cmed["label_id"])
cmed_test, cmed_valid = train_test_split(cmed_remaining, test_size=0.5, random_state=RANDOM_SEED, stratify=cmed_remaining["label_id"])
cmed_splits = {"train": cmed_train.reset_index(drop=True),
               "validation": cmed_valid.reset_index(drop=True),
               "test": cmed_test.reset_index(drop=True)}

dataset_splits = {"GoEmotions": goemotion_splits, "CMED": cmed_splits}
for dataset_name, splits in dataset_splits.items():
    print(f"\n{dataset_name} sizes:", {split: len(frame) for split, frame in splits.items()})
    print(splits["train"]["category"].value_counts().reindex(CATEGORY_LABELS, fill_value=0))

print("\nMYBully final-test examples:", len(mybully_test))
print(mybully_test["category"].value_counts().reindex(CATEGORY_LABELS, fill_value=0))

GoEmotions/train: removed 615 fear examples
GoEmotions/validation: removed 85 fear examples
GoEmotions/test: removed 90 fear examples


/home/jay/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MYBully/train: removed 61 fear examples
MYBully/source_validation: removed 19 fear examples
CMED: unmapped source labels omitted: ['關切語調']
CMED: removed 0 fear examples

GoEmotions sizes: {'train': 42795, 'validation': 5341, 'test': 5337}
category
anger        5336
disgust       580
joy         16948
sadness      2619
surprise     4489
neutral     12823
Name: count, dtype: int64

MYBully sizes: {'train': 3449, 'validation': 576, 'test': 576}
category
anger       1830
disgust      310
joy          324
sadness      207
surprise      47
neutral      731
Name: count, dtype: int64

CMED sizes: {'train': 2879, 'validation': 360, 'test': 360}
category
anger       422
disgust     323
joy         473
sadness     389
surprise    708
neutral     564
Name: count, dtype: int64


## Episodic prototype learning

Each episode draws support and query examples from one dataset, while episodes across all three datasets train the same encoder. Class prototypes are the mean support embeddings, and query examples are classified by distance to those prototypes.

In [5]:
"""Tokenizer and episodic sampling configuration."""

from transformers import AutoTokenizer, AutoModel
from torch.nn.utils.rnn import pad_sequence

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, timeout=120)

N_WAY = 5
N_SHOT = 2
N_QUERY = 3
TOKEN_CACHE = {}
EPISODES_PER_BATCH = 4

def sample_episode(frame, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY):
    """Sample a balanced episode and remap its classes to local episode IDs."""
    available = []
    for label_id, group in frame.groupby("label_id"):
        if len(group) >= n_shot + n_query:
            available.append((int(label_id), group))

    episode_n_way = min(n_way, len(available))
    if episode_n_way < 2:
        raise ValueError("An episode needs at least two eligible classes.")

    chosen = random.sample(available, episode_n_way)
    support_rows, query_rows = [], []
    for local_id, (_, group) in enumerate(chosen):
        rows = group.sample(n=n_shot + n_query, random_state=random.randint(0, 2**31 - 1))
        support_rows.extend((str(row.text), local_id) for row in rows.iloc[:n_shot].itertuples())
        query_rows.extend((str(row.text), local_id) for row in rows.iloc[n_shot:].itertuples())

    episode_category_ids = [label_id for label_id, _ in chosen]
    return support_rows, query_rows, episode_category_ids

def encode_texts(texts):
    missing = list(dict.fromkeys(text for text in texts if text not in TOKEN_CACHE))
    if missing:
        new_tokens = tokenizer(missing, max_length=MAX_LENGTH, truncation=True, padding=False)
        for text, ids, mask in zip(missing, new_tokens["input_ids"], new_tokens["attention_mask"]):
            TOKEN_CACHE[text] = {"input_ids": ids, "attention_mask": mask}

    encoded = tokenizer.pad(
        [TOKEN_CACHE[text] for text in texts],
        padding=True, pad_to_multiple_of=8, return_tensors="pt",
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}

print("Episode shape:", f"{N_WAY}-way, {N_SHOT}-shot, {N_QUERY}-query per class")

Episode shape: 5-way, 2-shot, 3-query per class


In [6]:
"""Prototype network and episode-level objective."""

class PrototypeNetwork(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state.float()
        mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
        mean_embedding = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        dropped_embedding = self.dropout(mean_embedding)
        return torch.nn.functional.normalize(dropped_embedding, p=2, dim=1)

def load_and_average_encoder_checkpoints(encoder, checkpoint_paths):
    """Average matching XLM-R encoder weights from the two classifier checkpoints."""
    target_state = encoder.state_dict()
    source_states = []

    for checkpoint_path in checkpoint_paths:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]

        encoder_state = {}
        for key, value in checkpoint.items():
            clean_key = key[7:] if key.startswith("module.") else key
            if clean_key.startswith("roberta."):
                clean_key = clean_key[len("roberta."):]
            if clean_key in target_state and target_state[clean_key].shape == value.shape:
                encoder_state[clean_key] = value

        coverage = len(encoder_state) / max(1, len(target_state))
        if coverage < 0.9:
            raise ValueError(
                f"Checkpoint {checkpoint_path} matched only {coverage:.1%} of encoder weights."
            )
        print(f"Loaded {len(encoder_state)} encoder tensors from {checkpoint_path}")
        source_states.append(encoder_state)

    shared_keys = set(source_states[0]) & set(source_states[1])
    shared_coverage = len(shared_keys) / max(1, len(target_state))
    if shared_coverage < 0.9:
        raise ValueError(f"The two checkpoints share only {shared_coverage:.1%} of encoder tensors.")

    averaged_state = dict(target_state)
    for key in shared_keys:
        first, second = source_states[0][key], source_states[1][key]
        if first.is_floating_point():
            averaged_state[key] = (first.float() + second.float()).div(2).to(first.dtype)
        else:
            averaged_state[key] = first

    encoder.load_state_dict(averaged_state, strict=True)

def initialize_encoder(model):
    base_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    for path in (GOEMOTIONS_CHECKPOINT, CMED_CHECKPOINT):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Required initialization checkpoint not found: {path}")
    load_and_average_encoder_checkpoints(
        base_model.encoder, [GOEMOTIONS_CHECKPOINT, CMED_CHECKPOINT]
    )

def prototypical_loss(model, support_rows, query_rows):
    support_texts, support_labels = zip(*support_rows)
    query_texts, query_labels = zip(*query_rows)

    all_inputs = encode_texts(list(support_texts) + list(query_texts))
    support_count = len(support_texts)
    support_targets = torch.tensor(support_labels, dtype=torch.long, device=DEVICE)
    query_targets = torch.tensor(query_labels, dtype=torch.long, device=DEVICE)

    embeddings = model(**all_inputs)
    support_embeddings = embeddings[:support_count]
    query_embeddings = embeddings[support_count:]
    episode_n_way = int(torch.cat([support_targets, query_targets]).max().item()) + 1
    prototypes = torch.stack([
        support_embeddings[support_targets == class_id].mean(dim=0)
        for class_id in range(episode_n_way)
    ])

    # Negative squared Euclidean distance acts as prototype-class logits.
    distances = torch.cdist(query_embeddings, prototypes, p=2).pow(2)
    logits = -distances
    loss = torch.nn.functional.cross_entropy(logits, query_targets)
    predictions = logits.argmax(dim=1)
    accuracy = (predictions == query_targets).float().mean()
    return loss, accuracy, predictions, query_targets

def batched_prototypical_loss(model, episode_batch):
    """Encode several episodes together to keep both GPUs occupied."""
    all_texts, layouts = [], []
    for support_rows, query_rows, _ in episode_batch:
        start = len(all_texts)
        support_texts, support_labels = zip(*support_rows)
        query_texts, query_labels = zip(*query_rows)
        all_texts.extend(support_texts)
        all_texts.extend(query_texts)
        layouts.append((
            start, len(support_texts),
            torch.tensor(support_labels, dtype=torch.long, device=DEVICE),
            torch.tensor(query_labels, dtype=torch.long, device=DEVICE),
        ))

    embeddings = model(**encode_texts(all_texts))
    losses, accuracies = [], []
    for start, support_count, support_targets, query_targets in layouts:
        query_start = start + support_count
        query_count = len(query_targets)
        support_embeddings = embeddings[start:query_start]
        query_embeddings = embeddings[query_start:query_start + query_count]
        n_way = int(torch.cat([support_targets, query_targets]).max().item()) + 1
        prototypes = torch.stack([
            support_embeddings[support_targets == class_id].mean(dim=0)
            for class_id in range(n_way)
        ])
        logits = -torch.cdist(query_embeddings, prototypes, p=2).pow(2)
        losses.append(torch.nn.functional.cross_entropy(logits, query_targets))
        accuracies.append((logits.argmax(dim=1) == query_targets).float().mean())

    return torch.stack(losses).mean(), torch.stack(accuracies).mean()

import os

model = PrototypeNetwork(MODEL_NAME).to(DEVICE)
initialize_encoder(model)
if NUM_GPUS > 1:
    model = torch.nn.DataParallel(model, device_ids=list(range(NUM_GPUS)))

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
GRAD_SCALER = torch.cuda.amp.GradScaler(enabled=USE_AMP and AMP_DTYPE == torch.float16)
print("Prototype encoder initialized on", DEVICE)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21025.93it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prototype encoder initialized on cuda:0


In [7]:
"""Episodic training and validation across all datasets."""

def run_episodes(model, optimizer=None, episodes_per_dataset=TRAIN_EPISODES_PER_DATASET):
    training = optimizer is not None
    model.train(training)
    episode_losses, episode_accuracies = [], []

    for dataset_name, splits in dataset_splits.items():
        frame = splits["train" if training else "validation"]
        for start in range(0, episodes_per_dataset, EPISODES_PER_BATCH):
            episode_batch = [
                sample_episode(frame)
                for _ in range(min(EPISODES_PER_BATCH, episodes_per_dataset - start))
            ]

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss, accuracy = batched_prototypical_loss(model, episode_batch)
                GRAD_SCALER.scale(loss).backward()
                GRAD_SCALER.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                GRAD_SCALER.step(optimizer)
                GRAD_SCALER.update()
            else:
                with torch.no_grad():
                    loss, accuracy = batched_prototypical_loss(model, episode_batch)

            episode_losses.append(float(loss.detach().cpu()))
            episode_accuracies.append(float(accuracy.detach().cpu()))

    return {
        "loss": float(np.mean(episode_losses)),
        "accuracy": float(np.mean(episode_accuracies)),
    }

best_validation_accuracy = -1.0
best_model_path = "best_prototypical_meta_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_episodes(
        model, optimizer, TRAIN_EPISODES_PER_DATASET
    )
    validation_metrics = run_episodes(
        model, None, VALIDATION_EPISODES_PER_DATASET
    )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f}, "
        f"train episode accuracy {train_metrics['accuracy']:.4f} | "
        f"validation loss {validation_metrics['loss']:.4f}, "
        f"validation episode accuracy {validation_metrics['accuracy']:.4f}"
    )

    if validation_metrics["accuracy"] > best_validation_accuracy:
        best_validation_accuracy = validation_metrics["accuracy"]
        base_model = model.module if isinstance(model, torch.nn.DataParallel) else model
        torch.save(base_model.state_dict(), best_model_path)
        print("Saved improved model to", best_model_path)

Epoch 1/5 | train loss 39.1786, train episode accuracy 0.1989 | validation loss 1.6092, validation episode accuracy 0.2889
Saved improved model to best_prototypical_meta_model.pt
Epoch 2/5 | train loss 37.3396, train episode accuracy 0.2061 | validation loss 1.6093, validation episode accuracy 0.3037
Saved improved model to best_prototypical_meta_model.pt
Epoch 3/5 | train loss 36.0315, train episode accuracy 0.2167 | validation loss 1.6093, validation episode accuracy 0.2685
Epoch 4/5 | train loss 37.0053, train episode accuracy 0.2067 | validation loss 1.6093, validation episode accuracy 0.2722
Epoch 5/5 | train loss 38.0180, train episode accuracy 0.1883 | validation loss 1.6093, validation episode accuracy 0.3000


In [8]:
"""Held-out test episodes and per-dataset classification reports."""

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Load the checkpoint selected by validation episode accuracy.
model_state = torch.load(best_model_path, map_location=DEVICE)
base_model = model.module if isinstance(model, torch.nn.DataParallel) else model
base_model.load_state_dict(model_state)

def evaluate_test_episodes(model, frame, n_episodes=VALIDATION_EPISODES_PER_DATASET):
    model.eval()
    actual_labels, predicted_labels = [], []

    with torch.no_grad():
        for _ in range(n_episodes):
            support_rows, query_rows, episode_category_ids = sample_episode(frame)
            _, _, predictions, targets = prototypical_loss(
                model, support_rows, query_rows
            )

            # Convert episode-local IDs back to the shared emotion IDs.
            local_predictions = predictions.cpu().tolist()
            local_targets = targets.cpu().tolist()
            predicted_labels.extend(episode_category_ids[i] for i in local_predictions)
            actual_labels.extend(episode_category_ids[i] for i in local_targets)

    return actual_labels, predicted_labels

for dataset_name, test_frame in [("MYBully", mybully_test)]:
    y_true, y_pred = evaluate_test_episodes(
        model, test_frame, MYBULLY_TEST_EPISODES
    )

    observed_ids = sorted(set(y_true) | set(y_pred))
    observed_names = [id2category[label_id] for label_id in observed_ids]

    print(f"\n==============================")
    print(f"{dataset_name.upper()} HELD-OUT EPISODE RESULTS")
    print(f"==============================")
    print(f"Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"Macro-F1:    {f1_score(y_true, y_pred, labels=observed_ids, average='macro', zero_division=0):.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred, labels=observed_ids, target_names=observed_names,
        digits=4, zero_division=0
    ))

    matrix = confusion_matrix(y_true, y_pred, labels=observed_ids)
    print("Confusion Matrix:")
    print(pd.DataFrame(matrix, index=observed_names, columns=observed_names))

/tmp/ipykernel_1482442/1845359257.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state = torch.load(best_model_path, map_location=DEVICE)



GOEMOTIONS HELD-OUT EPISODE RESULTS
Accuracy:    0.2056
Macro-F1:    0.2046

Classification Report:
              precision    recall  f1-score   support

       anger     0.1613    0.1667    0.1639        30
     disgust     0.2593    0.2121    0.2333        33
         joy     0.2069    0.2222    0.2143        27
     sadness     0.2188    0.2333    0.2258        30
    surprise     0.1333    0.1481    0.1404        27
     neutral     0.2581    0.2424    0.2500        33

    accuracy                         0.2056       180
   macro avg     0.2063    0.2042    0.2046       180
weighted avg     0.2092    0.2056    0.2068       180

Confusion Matrix:
          anger  disgust  joy  sadness  surprise  neutral
anger         5        3    1        7         9        5
disgust       4        7    7        4         5        6
joy           5        3    6        5         5        3
sadness       4        4    9        7         3        3
surprise      6        5    2        4         4